# Qwen3.5-9B on MMMU-Pro — Visual-Premise Diversity vs Correctness (v6, truncation removed)

**Experiment.** 24 MMMU-Pro questions (one per subject, all baseline *failures*). The model emits a single **visual premise** (one fact read from the image), **thinking ON**, **4096-token cap**, across 5 top_p × 16 samples.

**This notebook uses the CLEAN set only:** the 232/1920 samples that hit the 4096-token cap (`finish_reason='length'`) are **removed everywhere**. What remains: **1688 samples**, mean **14.1 per (example, top_p) cell** (range 4–16; 5 cells < 8).

A *cell* = one (example, top_p) pair → **120 cells**. **Vendi** = effective # of distinct premises; **frac_correct** = fraction judged correct against the image.

**Runtime → Run all.** Data embedded below; no uploads.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

# ---- embedded CLEAN per-cell data (120 cells, truncated samples removed) ----
DATA_JSON = r"""[{"id": "test_Accounting_55", "subject": "Accounting", "top_p": 0.5, "n": 16, "vendi": 2.3146, "cos": 0.2011, "fc": 1.0, "tokens": 1353.4}, {"id": "test_Accounting_55", "subject": "Accounting", "top_p": 0.7, "n": 16, "vendi": 3.1969, "cos": 0.308, "fc": 1.0, "tokens": 1541.5}, {"id": "test_Accounting_55", "subject": "Accounting", "top_p": 0.9, "n": 16, "vendi": 3.1142, "cos": 0.2832, "fc": 1.0, "tokens": 1719.1}, {"id": "test_Accounting_55", "subject": "Accounting", "top_p": 0.95, "n": 15, "vendi": 3.4041, "cos": 0.3242, "fc": 1.0, "tokens": 1562.5}, {"id": "test_Accounting_55", "subject": "Accounting", "top_p": 1.0, "n": 15, "vendi": 3.4291, "cos": 0.3176, "fc": 1.0, "tokens": 1860.2}, {"id": "test_Agriculture_19", "subject": "Agriculture", "top_p": 0.5, "n": 16, "vendi": 2.1503, "cos": 0.1594, "fc": 0.875, "tokens": 882.2}, {"id": "test_Agriculture_19", "subject": "Agriculture", "top_p": 0.7, "n": 16, "vendi": 2.0308, "cos": 0.1463, "fc": 0.9375, "tokens": 723.1}, {"id": "test_Agriculture_19", "subject": "Agriculture", "top_p": 0.9, "n": 16, "vendi": 2.2275, "cos": 0.1677, "fc": 0.9375, "tokens": 1093.8}, {"id": "test_Agriculture_19", "subject": "Agriculture", "top_p": 0.95, "n": 16, "vendi": 2.6744, "cos": 0.212, "fc": 0.875, "tokens": 910.8}, {"id": "test_Agriculture_19", "subject": "Agriculture", "top_p": 1.0, "n": 16, "vendi": 2.7569, "cos": 0.233, "fc": 0.8125, "tokens": 1422.8}, {"id": "test_Art_139", "subject": "Art", "top_p": 0.5, "n": 13, "vendi": 2.8434, "cos": 0.2526, "fc": 0.6923, "tokens": 1674.0}, {"id": "test_Art_139", "subject": "Art", "top_p": 0.7, "n": 12, "vendi": 3.3442, "cos": 0.3136, "fc": 0.6667, "tokens": 1547.1}, {"id": "test_Art_139", "subject": "Art", "top_p": 0.9, "n": 14, "vendi": 4.7532, "cos": 0.4746, "fc": 0.3571, "tokens": 1324.7}, {"id": "test_Art_139", "subject": "Art", "top_p": 0.95, "n": 15, "vendi": 3.8975, "cos": 0.348, "fc": 0.6, "tokens": 1577.7}, {"id": "test_Art_139", "subject": "Art", "top_p": 1.0, "n": 15, "vendi": 4.6276, "cos": 0.4416, "fc": 0.4, "tokens": 1963.6}, {"id": "test_Art_Theory_168", "subject": "Art_Theory", "top_p": 0.5, "n": 15, "vendi": 2.6759, "cos": 0.2363, "fc": 0.9333, "tokens": 1500.5}, {"id": "test_Art_Theory_168", "subject": "Art_Theory", "top_p": 0.7, "n": 13, "vendi": 2.1563, "cos": 0.1756, "fc": 0.9231, "tokens": 1092.7}, {"id": "test_Art_Theory_168", "subject": "Art_Theory", "top_p": 0.9, "n": 13, "vendi": 2.7475, "cos": 0.2447, "fc": 0.6923, "tokens": 1081.3}, {"id": "test_Art_Theory_168", "subject": "Art_Theory", "top_p": 0.95, "n": 16, "vendi": 2.7103, "cos": 0.2348, "fc": 0.75, "tokens": 1249.4}, {"id": "test_Art_Theory_168", "subject": "Art_Theory", "top_p": 1.0, "n": 13, "vendi": 2.3755, "cos": 0.2001, "fc": 0.6923, "tokens": 1655.6}, {"id": "test_Basic_Medical_Science_209", "subject": "Basic_Medical_Science", "top_p": 0.5, "n": 6, "vendi": 2.0359, "cos": 0.2128, "fc": 0.1667, "tokens": 1302.7}, {"id": "test_Basic_Medical_Science_209", "subject": "Basic_Medical_Science", "top_p": 0.7, "n": 10, "vendi": 2.2224, "cos": 0.1953, "fc": 0.1, "tokens": 2085.4}, {"id": "test_Basic_Medical_Science_209", "subject": "Basic_Medical_Science", "top_p": 0.9, "n": 11, "vendi": 1.9631, "cos": 0.1524, "fc": 0.1818, "tokens": 1252.8}, {"id": "test_Basic_Medical_Science_209", "subject": "Basic_Medical_Science", "top_p": 0.95, "n": 10, "vendi": 2.5163, "cos": 0.2409, "fc": 0.0, "tokens": 1757.3}, {"id": "test_Basic_Medical_Science_209", "subject": "Basic_Medical_Science", "top_p": 1.0, "n": 10, "vendi": 2.2179, "cos": 0.1901, "fc": 0.1, "tokens": 2167.7}, {"id": "test_Biology_132", "subject": "Biology", "top_p": 0.5, "n": 14, "vendi": 3.5054, "cos": 0.3253, "fc": 0.1429, "tokens": 1492.4}, {"id": "test_Biology_132", "subject": "Biology", "top_p": 0.7, "n": 11, "vendi": 2.84, "cos": 0.2745, "fc": 0.0909, "tokens": 1544.9}, {"id": "test_Biology_132", "subject": "Biology", "top_p": 0.9, "n": 12, "vendi": 2.7213, "cos": 0.2679, "fc": 0.0833, "tokens": 1248.6}, {"id": "test_Biology_132", "subject": "Biology", "top_p": 0.95, "n": 12, "vendi": 2.9945, "cos": 0.2785, "fc": 0.0833, "tokens": 1803.8}, {"id": "test_Biology_132", "subject": "Biology", "top_p": 1.0, "n": 15, "vendi": 3.1833, "cos": 0.281, "fc": 0.0667, "tokens": 1815.9}, {"id": "test_Chemistry_447", "subject": "Chemistry", "top_p": 0.5, "n": 16, "vendi": 1.592, "cos": 0.0896, "fc": 1.0, "tokens": 673.5}, {"id": "test_Chemistry_447", "subject": "Chemistry", "top_p": 0.7, "n": 16, "vendi": 1.8578, "cos": 0.1267, "fc": 1.0, "tokens": 741.9}, {"id": "test_Chemistry_447", "subject": "Chemistry", "top_p": 0.9, "n": 16, "vendi": 1.9235, "cos": 0.1318, "fc": 1.0, "tokens": 820.4}, {"id": "test_Chemistry_447", "subject": "Chemistry", "top_p": 0.95, "n": 15, "vendi": 2.0494, "cos": 0.1478, "fc": 1.0, "tokens": 791.5}, {"id": "test_Chemistry_447", "subject": "Chemistry", "top_p": 1.0, "n": 16, "vendi": 2.1926, "cos": 0.1619, "fc": 1.0, "tokens": 941.7}, {"id": "test_Clinical_Medicine_114", "subject": "Clinical_Medicine", "top_p": 0.5, "n": 16, "vendi": 1.9257, "cos": 0.1349, "fc": 1.0, "tokens": 998.4}, {"id": "test_Clinical_Medicine_114", "subject": "Clinical_Medicine", "top_p": 0.7, "n": 16, "vendi": 1.8578, "cos": 0.1209, "fc": 1.0, "tokens": 919.6}, {"id": "test_Clinical_Medicine_114", "subject": "Clinical_Medicine", "top_p": 0.9, "n": 16, "vendi": 2.057, "cos": 0.1448, "fc": 1.0, "tokens": 770.0}, {"id": "test_Clinical_Medicine_114", "subject": "Clinical_Medicine", "top_p": 0.95, "n": 16, "vendi": 1.8821, "cos": 0.1234, "fc": 1.0, "tokens": 1009.4}, {"id": "test_Clinical_Medicine_114", "subject": "Clinical_Medicine", "top_p": 1.0, "n": 14, "vendi": 2.3515, "cos": 0.1866, "fc": 1.0, "tokens": 1421.4}, {"id": "test_Computer_Science_226", "subject": "Computer_Science", "top_p": 0.5, "n": 16, "vendi": 1.7735, "cos": 0.125, "fc": 1.0, "tokens": 400.4}, {"id": "test_Computer_Science_226", "subject": "Computer_Science", "top_p": 0.7, "n": 16, "vendi": 2.2204, "cos": 0.1792, "fc": 1.0, "tokens": 663.1}, {"id": "test_Computer_Science_226", "subject": "Computer_Science", "top_p": 0.9, "n": 16, "vendi": 2.5112, "cos": 0.2047, "fc": 1.0, "tokens": 515.7}, {"id": "test_Computer_Science_226", "subject": "Computer_Science", "top_p": 0.95, "n": 16, "vendi": 2.3636, "cos": 0.1949, "fc": 1.0, "tokens": 646.7}, {"id": "test_Computer_Science_226", "subject": "Computer_Science", "top_p": 1.0, "n": 15, "vendi": 2.9238, "cos": 0.2562, "fc": 1.0, "tokens": 824.5}, {"id": "test_Design_135", "subject": "Design", "top_p": 0.5, "n": 14, "vendi": 4.4934, "cos": 0.4844, "fc": 1.0, "tokens": 1602.2}, {"id": "test_Design_135", "subject": "Design", "top_p": 0.7, "n": 14, "vendi": 3.9715, "cos": 0.3889, "fc": 1.0, "tokens": 1447.7}, {"id": "test_Design_135", "subject": "Design", "top_p": 0.9, "n": 14, "vendi": 4.5106, "cos": 0.4677, "fc": 1.0, "tokens": 1520.1}, {"id": "test_Design_135", "subject": "Design", "top_p": 0.95, "n": 14, "vendi": 4.1692, "cos": 0.4171, "fc": 0.8571, "tokens": 1514.3}, {"id": "test_Design_135", "subject": "Design", "top_p": 1.0, "n": 16, "vendi": 5.4589, "cos": 0.5017, "fc": 0.9375, "tokens": 1694.2}, {"id": "test_Diagnostics_and_Laboratory_Medicine_116", "subject": "Diagnostics_and_Laboratory_Medicine", "top_p": 0.5, "n": 16, "vendi": 2.6195, "cos": 0.2282, "fc": 0.0, "tokens": 879.2}, {"id": "test_Diagnostics_and_Laboratory_Medicine_116", "subject": "Diagnostics_and_Laboratory_Medicine", "top_p": 0.7, "n": 14, "vendi": 3.1257, "cos": 0.3056, "fc": 0.0, "tokens": 1060.1}, {"id": "test_Diagnostics_and_Laboratory_Medicine_116", "subject": "Diagnostics_and_Laboratory_Medicine", "top_p": 0.9, "n": 15, "vendi": 3.4383, "cos": 0.3087, "fc": 0.0, "tokens": 1168.7}, {"id": "test_Diagnostics_and_Laboratory_Medicine_116", "subject": "Diagnostics_and_Laboratory_Medicine", "top_p": 0.95, "n": 14, "vendi": 3.4242, "cos": 0.3143, "fc": 0.0, "tokens": 1114.4}, {"id": "test_Diagnostics_and_Laboratory_Medicine_116", "subject": "Diagnostics_and_Laboratory_Medicine", "top_p": 1.0, "n": 16, "vendi": 3.8953, "cos": 0.3619, "fc": 0.0, "tokens": 1254.9}, {"id": "test_Geography_144", "subject": "Geography", "top_p": 0.5, "n": 14, "vendi": 2.7432, "cos": 0.2431, "fc": 0.8571, "tokens": 2083.1}, {"id": "test_Geography_144", "subject": "Geography", "top_p": 0.7, "n": 14, "vendi": 2.5083, "cos": 0.2193, "fc": 1.0, "tokens": 1894.6}, {"id": "test_Geography_144", "subject": "Geography", "top_p": 0.9, "n": 15, "vendi": 2.6693, "cos": 0.2323, "fc": 0.8667, "tokens": 1352.8}, {"id": "test_Geography_144", "subject": "Geography", "top_p": 0.95, "n": 13, "vendi": 2.6898, "cos": 0.2338, "fc": 0.8462, "tokens": 1565.8}, {"id": "test_Geography_144", "subject": "Geography", "top_p": 1.0, "n": 16, "vendi": 2.7401, "cos": 0.2266, "fc": 0.8125, "tokens": 1547.8}, {"id": "test_History_100", "subject": "History", "top_p": 0.5, "n": 16, "vendi": 1.5833, "cos": 0.0891, "fc": 1.0, "tokens": 1341.6}, {"id": "test_History_100", "subject": "History", "top_p": 0.7, "n": 15, "vendi": 1.8241, "cos": 0.1298, "fc": 1.0, "tokens": 1470.3}, {"id": "test_History_100", "subject": "History", "top_p": 0.9, "n": 16, "vendi": 1.7688, "cos": 0.1114, "fc": 1.0, "tokens": 1434.6}, {"id": "test_History_100", "subject": "History", "top_p": 0.95, "n": 16, "vendi": 2.0085, "cos": 0.1547, "fc": 1.0, "tokens": 1471.2}, {"id": "test_History_100", "subject": "History", "top_p": 1.0, "n": 16, "vendi": 1.8986, "cos": 0.1308, "fc": 1.0, "tokens": 1512.9}, {"id": "test_Literature_11", "subject": "Literature", "top_p": 0.5, "n": 16, "vendi": 2.1426, "cos": 0.1627, "fc": 1.0, "tokens": 616.8}, {"id": "test_Literature_11", "subject": "Literature", "top_p": 0.7, "n": 16, "vendi": 2.3469, "cos": 0.1815, "fc": 1.0, "tokens": 686.6}, {"id": "test_Literature_11", "subject": "Literature", "top_p": 0.9, "n": 16, "vendi": 2.4307, "cos": 0.2004, "fc": 1.0, "tokens": 658.6}, {"id": "test_Literature_11", "subject": "Literature", "top_p": 0.95, "n": 16, "vendi": 2.5457, "cos": 0.2035, "fc": 1.0, "tokens": 652.0}, {"id": "test_Literature_11", "subject": "Literature", "top_p": 1.0, "n": 16, "vendi": 2.7736, "cos": 0.2324, "fc": 1.0, "tokens": 824.8}, {"id": "test_Manage_10", "subject": "Manage", "top_p": 0.5, "n": 16, "vendi": 1.7348, "cos": 0.1146, "fc": 1.0, "tokens": 538.8}, {"id": "test_Manage_10", "subject": "Manage", "top_p": 0.7, "n": 16, "vendi": 1.6438, "cos": 0.0975, "fc": 1.0, "tokens": 559.6}, {"id": "test_Manage_10", "subject": "Manage", "top_p": 0.9, "n": 16, "vendi": 1.8978, "cos": 0.1331, "fc": 1.0, "tokens": 607.4}, {"id": "test_Manage_10", "subject": "Manage", "top_p": 0.95, "n": 16, "vendi": 1.6905, "cos": 0.1018, "fc": 1.0, "tokens": 573.9}, {"id": "test_Manage_10", "subject": "Manage", "top_p": 1.0, "n": 16, "vendi": 1.9243, "cos": 0.1321, "fc": 1.0, "tokens": 581.3}, {"id": "test_Marketing_161", "subject": "Marketing", "top_p": 0.5, "n": 15, "vendi": 2.3707, "cos": 0.1889, "fc": 0.4667, "tokens": 1458.9}, {"id": "test_Marketing_161", "subject": "Marketing", "top_p": 0.7, "n": 14, "vendi": 2.5469, "cos": 0.2147, "fc": 0.6429, "tokens": 1430.1}, {"id": "test_Marketing_161", "subject": "Marketing", "top_p": 0.9, "n": 16, "vendi": 2.5517, "cos": 0.2141, "fc": 0.8125, "tokens": 1449.1}, {"id": "test_Marketing_161", "subject": "Marketing", "top_p": 0.95, "n": 16, "vendi": 2.7271, "cos": 0.2349, "fc": 0.75, "tokens": 1749.4}, {"id": "test_Marketing_161", "subject": "Marketing", "top_p": 1.0, "n": 16, "vendi": 3.1005, "cos": 0.2682, "fc": 0.75, "tokens": 1645.9}, {"id": "test_Materials_165", "subject": "Materials", "top_p": 0.5, "n": 16, "vendi": 2.2815, "cos": 0.2215, "fc": 1.0, "tokens": 943.9}, {"id": "test_Materials_165", "subject": "Materials", "top_p": 0.7, "n": 16, "vendi": 2.2133, "cos": 0.1762, "fc": 0.9375, "tokens": 1484.5}, {"id": "test_Materials_165", "subject": "Materials", "top_p": 0.9, "n": 16, "vendi": 2.7367, "cos": 0.2493, "fc": 1.0, "tokens": 1176.8}, {"id": "test_Materials_165", "subject": "Materials", "top_p": 0.95, "n": 16, "vendi": 2.5198, "cos": 0.2114, "fc": 1.0, "tokens": 1372.5}, {"id": "test_Materials_165", "subject": "Materials", "top_p": 1.0, "n": 16, "vendi": 3.0679, "cos": 0.2781, "fc": 1.0, "tokens": 1253.2}, {"id": "test_Math_435", "subject": "Math", "top_p": 0.5, "n": 9, "vendi": 2.7641, "cos": 0.2778, "fc": 0.7778, "tokens": 1896.7}, {"id": "test_Math_435", "subject": "Math", "top_p": 0.7, "n": 9, "vendi": 2.7725, "cos": 0.2868, "fc": 0.6667, "tokens": 2247.8}, {"id": "test_Math_435", "subject": "Math", "top_p": 0.9, "n": 8, "vendi": 3.3618, "cos": 0.4072, "fc": 0.875, "tokens": 2221.8}, {"id": "test_Math_435", "subject": "Math", "top_p": 0.95, "n": 10, "vendi": 3.2518, "cos": 0.3354, "fc": 0.8, "tokens": 2074.1}, {"id": "test_Math_435", "subject": "Math", "top_p": 1.0, "n": 8, "vendi": 3.0096, "cos": 0.3281, "fc": 0.875, "tokens": 2332.0}, {"id": "test_Music_299", "subject": "Music", "top_p": 0.5, "n": 4, "vendi": 1.5515, "cos": 0.1366, "fc": 1.0, "tokens": 1970.5}, {"id": "test_Music_299", "subject": "Music", "top_p": 0.7, "n": 7, "vendi": 1.8401, "cos": 0.1788, "fc": 0.7143, "tokens": 2104.0}, {"id": "test_Music_299", "subject": "Music", "top_p": 0.9, "n": 8, "vendi": 2.6463, "cos": 0.2895, "fc": 1.0, "tokens": 2493.8}, {"id": "test_Music_299", "subject": "Music", "top_p": 0.95, "n": 8, "vendi": 2.2592, "cos": 0.2202, "fc": 0.875, "tokens": 2338.8}, {"id": "test_Music_299", "subject": "Music", "top_p": 1.0, "n": 5, "vendi": 2.3797, "cos": 0.3316, "fc": 0.6, "tokens": 1869.0}, {"id": "test_Pharmacy_201", "subject": "Pharmacy", "top_p": 0.5, "n": 7, "vendi": 1.653, "cos": 0.1213, "fc": 1.0, "tokens": 2796.7}, {"id": "test_Pharmacy_201", "subject": "Pharmacy", "top_p": 0.7, "n": 11, "vendi": 2.5288, "cos": 0.3151, "fc": 1.0, "tokens": 2222.3}, {"id": "test_Pharmacy_201", "subject": "Pharmacy", "top_p": 0.9, "n": 13, "vendi": 1.8681, "cos": 0.1396, "fc": 1.0, "tokens": 2418.8}, {"id": "test_Pharmacy_201", "subject": "Pharmacy", "top_p": 0.95, "n": 8, "vendi": 2.336, "cos": 0.272, "fc": 1.0, "tokens": 2084.6}, {"id": "test_Pharmacy_201", "subject": "Pharmacy", "top_p": 1.0, "n": 9, "vendi": 2.1612, "cos": 0.2081, "fc": 1.0, "tokens": 2556.3}, {"id": "test_Psychology_105", "subject": "Psychology", "top_p": 0.5, "n": 12, "vendi": 2.3831, "cos": 0.206, "fc": 0.9167, "tokens": 1540.8}, {"id": "test_Psychology_105", "subject": "Psychology", "top_p": 0.7, "n": 15, "vendi": 2.5609, "cos": 0.2088, "fc": 1.0, "tokens": 1328.1}, {"id": "test_Psychology_105", "subject": "Psychology", "top_p": 0.9, "n": 14, "vendi": 2.5363, "cos": 0.2109, "fc": 1.0, "tokens": 1432.7}, {"id": "test_Psychology_105", "subject": "Psychology", "top_p": 0.95, "n": 15, "vendi": 2.8061, "cos": 0.2368, "fc": 1.0, "tokens": 1442.1}, {"id": "test_Psychology_105", "subject": "Psychology", "top_p": 1.0, "n": 14, "vendi": 3.057, "cos": 0.2663, "fc": 1.0, "tokens": 1574.0}, {"id": "test_Public_Health_152", "subject": "Public_Health", "top_p": 0.5, "n": 16, "vendi": 2.9911, "cos": 0.2725, "fc": 1.0, "tokens": 1297.1}, {"id": "test_Public_Health_152", "subject": "Public_Health", "top_p": 0.7, "n": 15, "vendi": 3.7112, "cos": 0.3923, "fc": 0.9333, "tokens": 1540.1}, {"id": "test_Public_Health_152", "subject": "Public_Health", "top_p": 0.9, "n": 13, "vendi": 3.2952, "cos": 0.3345, "fc": 1.0, "tokens": 1979.9}, {"id": "test_Public_Health_152", "subject": "Public_Health", "top_p": 0.95, "n": 16, "vendi": 3.6623, "cos": 0.345, "fc": 0.9375, "tokens": 1839.6}, {"id": "test_Public_Health_152", "subject": "Public_Health", "top_p": 1.0, "n": 16, "vendi": 3.8563, "cos": 0.3813, "fc": 0.9375, "tokens": 1617.6}, {"id": "test_Sociology_102", "subject": "Sociology", "top_p": 0.5, "n": 16, "vendi": 2.0702, "cos": 0.1578, "fc": 1.0, "tokens": 980.1}, {"id": "test_Sociology_102", "subject": "Sociology", "top_p": 0.7, "n": 16, "vendi": 2.2068, "cos": 0.176, "fc": 1.0, "tokens": 1003.1}, {"id": "test_Sociology_102", "subject": "Sociology", "top_p": 0.9, "n": 16, "vendi": 2.1184, "cos": 0.1584, "fc": 1.0, "tokens": 1214.2}, {"id": "test_Sociology_102", "subject": "Sociology", "top_p": 0.95, "n": 16, "vendi": 2.4459, "cos": 0.1979, "fc": 1.0, "tokens": 1264.5}, {"id": "test_Sociology_102", "subject": "Sociology", "top_p": 1.0, "n": 15, "vendi": 2.8577, "cos": 0.2459, "fc": 1.0, "tokens": 1658.3}, {"id": "validation_Economics_26", "subject": "Economics", "top_p": 0.5, "n": 16, "vendi": 2.2529, "cos": 0.1735, "fc": 1.0, "tokens": 2013.2}, {"id": "validation_Economics_26", "subject": "Economics", "top_p": 0.7, "n": 16, "vendi": 2.1969, "cos": 0.1637, "fc": 0.9375, "tokens": 1731.7}, {"id": "validation_Economics_26", "subject": "Economics", "top_p": 0.9, "n": 16, "vendi": 2.1832, "cos": 0.1608, "fc": 1.0, "tokens": 1489.1}, {"id": "validation_Economics_26", "subject": "Economics", "top_p": 0.95, "n": 16, "vendi": 2.4082, "cos": 0.1866, "fc": 0.9375, "tokens": 1353.3}, {"id": "validation_Economics_26", "subject": "Economics", "top_p": 1.0, "n": 16, "vendi": 2.388, "cos": 0.1845, "fc": 1.0, "tokens": 1381.0}]"""
df = pd.DataFrame(json.loads(DATA_JSON))
print(df.shape, "cells | mean samples/cell:", round(df.n.mean(),1))
df.head()

## Correlation summary

`vendi <-> frac_correct` across cells (clean set), with and without the 3 image-hallucination questions.


In [ ]:
def corr(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = ~(np.isnan(x) | np.isnan(y)); x, y = x[m], y[m]
    if len(x) < 3 or x.std()==0 or y.std()==0: return np.nan, np.nan
    pear = float(np.corrcoef(x, y)[0,1])
    rx, ry = np.argsort(np.argsort(x)), np.argsort(np.argsort(y))
    return pear, float(np.corrcoef(rx, ry)[0,1])

DROP = {"test_Diagnostics_and_Laboratory_Medicine_116",
        "test_Basic_Medical_Science_209", "test_Biology_132"}

def summarize(d, label):
    vp, vs = corr(d.vendi, d.fc); cp, cs = corr(d["cos"], d.fc)
    return {"filter": label, "n_cells": int(d.fc.notna().sum()),
            "mean_acc": round(float(d.fc.mean()),3),
            "vendi~corr (Pearson)": round(vp,3), "vendi~corr (Spearman)": round(vs,3),
            "cos~corr (Pearson)": round(cp,3)}

pd.DataFrame([summarize(df, "clean (all 24 Q)"),
              summarize(df[~df.id.isin(DROP)], "clean minus 3 halluc. Q (21 Q)")])

## Core scatter — diversity vs correctness (clean set)

Each point is a cell. Color = top_p. Black line = OLS fit.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, col, name in [(axes[0],"vendi","Vendi score"), (axes[1],"cos","mean cosine distance")]:
    sc = ax.scatter(df[col], df.fc, c=df.top_p, cmap="viridis", s=45, edgecolor="k", linewidth=0.3, alpha=0.85)
    x, y = df[col].to_numpy(float), df.fc.to_numpy(float)
    m = ~(np.isnan(x)|np.isnan(y)); x,y=x[m],y[m]
    b,a = np.polyfit(x,y,1); xs=np.linspace(x.min(),x.max(),50); ax.plot(xs, a+b*xs, "k-", lw=2)
    p,s = corr(x,y)
    ax.set_title(f"{name} vs correctness\nPearson={p:+.3f}  Spearman={s:+.3f}")
    ax.set_xlabel(name); ax.set_ylabel("frac_correct")
fig.colorbar(sc, ax=axes, label="top_p", shrink=0.85); plt.show()

## Per-top_p — which top_p is optimal?

Harmonic mean H = 2·c·d/(c+d), with c=frac_correct and d=(vendi-1)/15 (normalized diversity).


In [ ]:
df["dnorm"] = (df.vendi - 1) / 15.0
df["harmonic"] = np.where(df.fc + df.dnorm == 0, 0.0, 2*df.fc*df.dnorm/(df.fc+df.dnorm))
g = df.groupby("top_p").agg(correctness=("fc","mean"), vendi=("vendi","mean"),
                            harmonic=("harmonic","mean")).reset_index()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].plot(g.top_p, g.correctness, "o-", color="green"); ax[0].set_title("correctness vs top_p"); ax[0].set_ylim(0,1)
ax[1].plot(g.top_p, g.vendi, "o-"); ax[1].set_title("Vendi diversity vs top_p")
ax[2].plot(g.top_p, g.harmonic, "o-", color="purple"); ax[2].set_title("harmonic mean vs top_p")
for a in ax: a.set_xlabel("top_p")
plt.tight_layout(); plt.show()
print("argmax correctness:", g.loc[g.correctness.idxmax(),"top_p"],
      "| argmax harmonic:", g.loc[g.harmonic.idxmax(),"top_p"])
g.round(3)

## Per-example breakdown (clean set)

Mean premise accuracy per question (over top_p), sorted. Low-accuracy = image-hallucination cases.


In [ ]:
pe = df.groupby(["id","subject"]).agg(acc=("fc","mean"), vendi=("vendi","mean")).reset_index().sort_values("acc")
fig, ax = plt.subplots(figsize=(9, 7)); y = np.arange(len(pe))
ax.barh(y, pe.acc, color=plt.cm.RdYlGn(pe.acc))
ax.set_yticks(y); ax.set_yticklabels(pe.subject, fontsize=8)
ax.set_xlabel("mean premise accuracy"); ax.set_xlim(0,1); ax.set_title("Per-example premise accuracy — clean set (24 questions)")
for i,v in enumerate(pe.vendi): ax.text(0.01, i, f"  vendi={v:.2f}", va="center", fontsize=7)
plt.tight_layout(); plt.show()

---
### Takeaways (clean set — truncation removed everywhere)
- **Diversity ↔ correctness is negative but moderate:** vendi~corr Pearson ≈ **−0.26** (−0.37 dropping the 3 hallucination Q). The stronger −0.51 seen with truncated samples included was largely an artifact.
- **Correctness slightly *declines* with top_p** (best at low top_p ≈0.5–0.9, ~0.82 → 0.79 at 1.0). The harmonic optimum at top_p=1.0 is degenerate (just chasing diversity).
- **Practical pick: top_p ≈ 0.5–0.7** for correct premises; go higher only if you want premise diversity for ensembling.
- Edit `DROP` to toggle the 3 outlier questions.
